# Ambient-A2A: EXAONE 4.0 1.2B SFT → DPO

이 노트북은 스마트홈 에이전트 데이터로 EXAONE 4.0 1.2B를 학습한다. 데이터는 OpenClaw가 실제로 보낸 요청(시스템 프롬프트, tool 정의, 대화)과 Haiku 5.5의 정답 응답이다.
1. **SFT**: Haiku의 응답을 그대로 따라 하도록 학습한다(tool 호출 형식, 한국어 답변).
2. **DPO**: 같은 요청에 대해 Haiku 응답(chosen)을 기본 EXAONE의 틀린 응답(rejected)보다 선호하도록 학습한다.
3. 학습한 모델을 GGUF(Q8_0)로 바꿔 내려받는다. 로컬 llama.cpp에서 같은 검사(`training/run_scenarios.py --set heldout`)로 평가한다.

**런타임**: 런타임 → 런타임 유형 변경 → GPU(T4 이상, 가능하면 L4/A100). **데이터**: `ambient-dpo-data.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install "trl==1.16.0" "peft>=0.17" "transformers>=4.56" "accelerate>=1.10" "datasets>=4.0"
!pip -q uninstall -y torchao   # Colab ships torchao 0.10; new PEFT refuses versions <= 0.16 and we don't use it

## 1. 데이터 올리기와 Drive 연결 (결과를 Drive에 저장해서 런타임이 끊겨도 남게 한다)

In [ ]:
import os, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ambient-dpo'
os.makedirs(OUT, exist_ok=True)
up = files.upload()            # ambient-dpo-data.zip 선택
zipfile.ZipFile(next(iter(up))).extractall('/content/data')
print(sorted(os.listdir('/content/data')))

## 2. 모델의 chat template으로 텍스트 만들기 (서빙할 때 llama.cpp가 쓰는 형식과 같게)

In [ ]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset

BASE = 'LGAI-EXAONE/EXAONE-4.0-1.2B'
tok = AutoTokenizer.from_pretrained(BASE)
EOT = '[|endofturn|]'
assert tok.eos_token == EOT, tok.eos_token

def load(name):
    return [json.loads(l) for l in open(f'/content/data/{name}.jsonl')]

def render(tools, prompt, answer=None):
    p = tok.apply_chat_template(prompt, tools=tools or None, add_generation_prompt=True, tokenize=False)
    if answer is None:
        return p
    full = tok.apply_chat_template(prompt + answer, tools=tools or None, tokenize=False)
    assert full.startswith(p), 'template prefix mismatch'
    c = full[len(p):].rstrip('\n')
    assert c.endswith(EOT), c[-40:]
    return c[: -len(EOT)]   # the trainer appends EOS itself

def sft_ds(split):
    return Dataset.from_list([{'prompt': render(r['tools'], r['prompt']),
                               'completion': render(r['tools'], r['prompt'], r['completion'])} for r in load(f'sft_{split}')])

def dpo_ds(split):
    return Dataset.from_list([{'prompt': render(r['tools'], r['prompt']),
                               'chosen': render(r['tools'], r['prompt'], r['chosen']),
                               'rejected': render(r['tools'], r['prompt'], r['rejected'])} for r in load(f'dpo_{split}')])

sft_train, sft_eval, dpo_train, dpo_eval = sft_ds('train'), sft_ds('eval'), dpo_ds('train'), dpo_ds('eval')
print(len(sft_train), len(sft_eval), len(dpo_train), len(dpo_eval))
lens = [len(tok(x['prompt'] + x['completion']).input_ids) for x in sft_train]
print('max tokens', max(lens), 'mean', sum(lens) // len(lens))
print(dpo_train[0]['chosen'][:300]); print('---'); print(dpo_train[0]['rejected'][:300])

## 3. SFT (LoRA)

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

bf16 = torch.cuda.is_bf16_supported(including_emulation=False)   # T4 only emulates bf16 (slow): use fp16 there
DTYPE = torch.bfloat16 if bf16 else torch.float16
MAXLEN = 8192
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
model = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE)

sft = SFTTrainer(
    model=model, processing_class=tok, peft_config=lora,
    train_dataset=sft_train, eval_dataset=sft_eval,
    args=SFTConfig(output_dir=f'{OUT}/sft', num_train_epochs=3, learning_rate=2e-4,
                   per_device_train_batch_size=1, gradient_accumulation_steps=8,
                   gradient_checkpointing=True, max_length=MAXLEN, bf16=bf16, fp16=not bf16,
                   logging_steps=5, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
                   completion_only_loss=True, report_to='none'))
ids = sft.train_dataset[0]['input_ids']   # exactly one EOS at the end of every example
assert ids[-1] == tok.eos_token_id and ids[-2] != tok.eos_token_id, tok.decode(ids[-5:])
sft.train()
sft_model = sft.model.merge_and_unload()
sft_model.save_pretrained(f'{OUT}/sft-merged'); tok.save_pretrained(f'{OUT}/sft-merged')

## 4. DPO (SFT 모델 위에 새 LoRA. 기준 모델은 LoRA를 끈 SFT 모델)

In [ ]:
from trl import DPOConfig, DPOTrainer

del sft, model; torch.cuda.empty_cache()
policy = AutoModelForCausalLM.from_pretrained(f'{OUT}/sft-merged', dtype=DTYPE)
dpo = DPOTrainer(
    model=policy, processing_class=tok, peft_config=lora,
    train_dataset=dpo_train, eval_dataset=dpo_eval,
    args=DPOConfig(output_dir=f'{OUT}/dpo', num_train_epochs=2, learning_rate=5e-6, beta=0.1,
                   per_device_train_batch_size=1, gradient_accumulation_steps=8,
                   gradient_checkpointing=True, max_length=MAXLEN, bf16=bf16, fp16=not bf16,
                   logging_steps=5, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
                   report_to='none'))
dpo.train()
final = dpo.model.merge_and_unload()
final.save_pretrained(f'{OUT}/final'); tok.save_pretrained(f'{OUT}/final')

## 5. 빠른 오프라인 확인: 평가 프롬프트에서 tool 호출이 정답(Haiku)과 같은 비율

In [ ]:
import re
def calls(text):
    out = []
    for m in re.finditer(r'<tool_call>(.*?)</tool_call>', text, re.S):
        try:
            j = json.loads(m.group(1)); a = j.get('arguments', {})
            out.append((j.get('name'), json.dumps({k: a.get(k) for k in ('target','device','command','value')}, sort_keys=True)))
        except Exception:
            out.append(('<bad json>', ''))
    return sorted(out)

def score(m, rows):
    m.eval(); hit = 0
    for r in rows:
        p = render(r['tools'], r['prompt'])
        x = tok(p, return_tensors='pt').to(m.device)
        y = m.generate(**x, max_new_tokens=400, do_sample=False)
        gen = tok.decode(y[0][x.input_ids.shape[1]:], skip_special_tokens=False)
        hit += calls(gen) == calls(render(r['tools'], r['prompt'], r['completion']))
    return hit / len(rows)

rows = [r for r in load('sft_eval') if r['completion'][0].get('tool_calls')]   # only answers that call tools
print(len(rows), 'eval rows with tool calls')
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE).to('cuda')
print('base  match:', score(base, rows)); del base
print('final match:', score(final.to('cuda'), rows))

## 6. GGUF(Q8_0)로 변환해 Drive에 저장하고 내려받기

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py {OUT}/final --outtype q8_0 --outfile {OUT}/EXAONE-4.0-1.2B-ambient-dpo-Q8_0.gguf
files.download(f'{OUT}/EXAONE-4.0-1.2B-ambient-dpo-Q8_0.gguf')